# QASPER — `token` vs `ahc` (Adaptive Hybrid Chunking)

A high-observability follow-up to the QuALITY pilot. We run **two arms** inside an otherwise **frozen** RAPTOR pipeline — only the leaf chunker changes — on the **same** QASPER papers, and ask the one question that matters: *does AHC move Answer-F1 vs baseline RAPTOR on the same documents and the same reader?*

| Arm | Chunker |
|---|---|
| `token` | original RAPTOR sentence/token splitter (**baseline RAPTOR**) |
| `ahc` | **Adaptive Hybrid Chunking** (structure score → route → repair) — *our method* |

- **Dataset:** QASPER (QA over NLP papers — structured documents with headings/sections).
- **Metric:** Answer token-F1 (the paper's QASPER metric).
- **Frozen pipeline:** same gpt-oss-120b for summarization **and** QA across both arms, same retrieval budget — so the comparison isolates the chunker.
- **Scale:** 5 docs, 1 seed, run on the **Cerebras free tier** (`gpt-oss-120b`).

> **Why QASPER, and why now.** This is the experiment that **follows up the QuALITY pilot** (documented in section 6). On QuALITY — plain narrative prose — AHC's structure score was 0 for every doc, so AHC silently fell back to token chunking and the arms tied. QASPER is the **structured-document** test where AHC's structure path can actually engage. Everything below is built for **observability**: a no-LLM routing diagnostic *before* we spend budget, verbose tqdm progress, and a persisted JSONL run-log + per-record dump.

## 1. Clone the repo and install the full stack

In [ ]:
!git clone -b ckraptor https://github.com/MissLostCodes/raptor.git
%cd raptor
!pip install -q -r requirements-colab.txt

## 2. Credentials + METEOR data

We use **Cerebras** as the inference provider — it serves the **same `gpt-oss-120b`** model on a genuinely free tier (**1,000,000 tokens/day, no credit card**), which is ~20× more usable than OpenRouter's free tier (50 requests/day without credits). Same open-weights model → results stay comparable.

Get a free key at **https://cloud.cerebras.ai** → *API Keys*. (Groq — https://console.groq.com — also serves `gpt-oss-120b` free and is a drop-in fallback, but its 200K tokens/day is ~5× smaller.)

In [ ]:
import os, getpass
# FREE, no credit card: https://cloud.cerebras.ai -> API Keys.
# Cerebras serves gpt-oss-120b free at 1M tokens/day (vs OpenRouter's 50 req/day).
from google.colab import userdata
os.environ["CEREBRAS_API_KEY"] = userdata.get("CEREBRAS_API_KEY")

import nltk
for pkg in ('wordnet', 'omw-1.4', 'punkt'):
    nltk.download(pkg, quiet=True)
print('ready')

## 2b. Persist cache + results to Google Drive (recommended)

Free-tier Colab sessions recycle, and free-tier rate limits mean a full run spans many sittings. Mounting Drive keeps `.llm_cache/` and `results/` across disconnects, so every rerun **resumes from cache** instead of re-spending LLM calls. Authorize Drive access when prompted. (Off Colab this falls back to a local `./raptor_runs`.) We also write this notebook's run-log + per-record dump under `RUN_DIR`.

In [ ]:
# Persist the LLM cache + results to Google Drive so a disconnect never loses
# the (expensive) cached calls. Re-running then resumes from cache.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    RUN_DIR = '/content/drive/MyDrive/raptor_runs'
except Exception as e:
    print('Not on Colab / Drive unavailable -> using local ./raptor_runs', e)
    RUN_DIR = 'raptor_runs'

import os
CACHE_DIR = os.path.join(RUN_DIR, '.llm_cache')
RESULTS_DIR = os.path.join(RUN_DIR, 'results')
os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
print('RUN_DIR   =', RUN_DIR)
print('CACHE_DIR =', CACHE_DIR)
print('RESULTS_DIR =', RESULTS_DIR)

## 3. (Optional) Smoke-test the harness offline
All logic is unit-tested without network/model loads; run it to confirm the clone is intact.

In [ ]:
!python -m pytest tests/ -q

## 4. Prior pilot: QuALITY (`token` vs `ahc`)

This QASPER run is the **direct follow-up** to a QuALITY pilot we ran with the identical two-arm setup (baseline RAPTOR vs AHC, same gpt-oss-120b reader). The pilot result:

| arm | metric | value | n | 95% CI |
|---|---|---:|---:|---|
| `ahc` | accuracy | 0.4000 | 105 | [0.3143, 0.5048] |
| `ahc` | accuracy_hard | 0.3729 | 59 | — |
| `token` | accuracy | 0.3810 | 105 | [0.2952, 0.4857] |
| `token` | accuracy_hard | 0.3559 | 59 | — |

- **AHC structure-routing rate = 0.0000** over 6 docs.
- **Dropped (moderation) = 4** questions (removed uniformly from both arms).

**Interpretation.** On QuALITY (plain narrative prose) AHC's structure score was **0 for every doc**, so AHC fell back to token chunking — the two arms were effectively **identical**, and the result is a **statistical tie** (≈42 vs 40 correct out of 105; CIs overlap almost entirely). That is not a failure of AHC so much as a property of the *input*: there was no structure to exploit, so the adaptive router correctly declined to engage the structure path.

**That is exactly why QASPER comes next.** QASPER papers carry explicit headings and sections, so AHC's structure path *can* engage here. The danger is that it engages **silently with 0 rate again** — so before spending any LLM budget we run a deterministic routing diagnostic (section 6) and **assert** the routing rate is non-zero.

## 5. Configure — QASPER, `token` vs `ahc`, 5 docs

`token` (baseline RAPTOR) vs `ahc` (our method) on **QASPER only**, **5 docs**, 1 seed. `leaf_max_tokens=100` and `retrieval_max_tokens=2000` are the paper-faithful settings — do not change them, or the numbers stop being comparable. Caching is keyed by `(model, prompt)`, so reruns are cheap and the run is resumable from Drive.

In [ ]:
from experiments.config import ExperimentConfig

# QASPER follow-up (free Cerebras tier) -- token (baseline RAPTOR) vs ahc (ours).
# Same open-weights gpt-oss-120b for summarization AND QA -> arm-vs-arm comparable.
cfg = ExperimentConfig(
    model='gpt-oss-120b',                       # Cerebras model id
    base_url='https://api.cerebras.ai/v1',      # OpenAI-compatible endpoint
    api_key_env='CEREBRAS_API_KEY',
    arms=['token', 'ahc'],                      # baseline RAPTOR vs our contribution
    datasets=['qasper'],                        # structured docs -> AHC can route to structure
    subset_sizes={'qasper': 5},                 # 5 papers, 1 sitting
    seeds=[0],
    retrieval_max_tokens=2000,                  # paper's collapsed-tree main setting
    leaf_max_tokens=100,                        # paper-faithful -- do NOT raise
    tau=0.5,                                    # AHC routing threshold
    cache_dir=CACHE_DIR,                        # on Drive (section 2b)
    results_dir=RESULTS_DIR,
)
cfg.to_dict()

## 6. Pre-run structure diagnostic (no LLM — the routing gate)

The QuALITY pilot **failed silently**: AHC's structure-routing rate was 0, so it quietly behaved like the `token` baseline and we only learned that *after* spending the run. Before spending **any** LLM budget here, we confirm AHC will actually route QASPER docs to the **structure** path.

This diagnostic is **deterministic, no-LLM, and cheap**: `structure_score(text)` looks only at cheap regex surface features (heading density, table-of-contents, heading-spacing regularity) and returns a scalar in `[0, 1]`. With `tau = 0.5`, a doc routes to **structure** when `score >= tau`, else falls back to **token**. We print every doc's score + full feature dict + predicted route, then compute the routing rate and **assert it is > 0** — failing loudly here is far cheaper than discovering a silent 0 after an hour of LLM calls.

In [ ]:
from experiments.datasets import get_loader
from raptor.chunking.structure_score import structure_score

# Load the SAME 5 QASPER docs the run will use (deterministic; no network LLM).
docs = get_loader('qasper').load(limit=cfg.subset_sizes['qasper'])
print(f'loaded {len(docs)} QASPER docs  |  tau = {cfg.tau}\n')

routes = []
for d in docs:
    score, feats = structure_score(d.text)
    route = 'structure' if score >= cfg.tau else 'token'
    routes.append(route)
    n_lines = sum(1 for ln in d.text.splitlines() if ln.strip())
    print(f'doc_id           : {d.doc_id}')
    print(f'  structure_score: {round(score, 4)}   -> route: {route.upper()}')
    print(f'  features       : {feats}')
    print(f'  non-empty lines: {n_lines}')
    print(f'  questions      : {len(d.questions)}')
    print()

route_rate = sum(r == 'structure' for r in routes) / len(routes) if routes else 0.0
print(f'AHC structure-routing rate (predicted): {route_rate:.4f}  '
      f'({sum(r == "structure" for r in routes)}/{len(routes)} docs)')
assert route_rate > 0, \
    'AHC never routes to structure on QASPER -- investigate before spending LLM budget.'
print('OK: AHC will engage the structure path on QASPER -> the run is worth its budget.')

## 7. Run
Builds one tree per (arm, doc) — SBERT embeddings + gpt-oss summaries — then answers every question with the same gpt-oss reader and scores Answer-F1.

**Robust by construction** — a long run is no longer a fragile black box:
- **Progress:** a `tqdm` bar tracks each `dataset/arm`, plus a per-dataset/arm line — and here we *also* tee every progress message to a persisted JSONL run-log.
- **Crash-safe + resumable:** results are checkpointed to the results file after *every doc*, and `resume=True` skips already-completed `(dataset, arm, doc)` on a rerun — so a Colab disconnect costs you at most one doc.
- **Moderation-proof:** if a question is rejected by the provider's moderation, it is recorded as `blocked` and the run continues — a single block never kills the grid.
- **Permanent vs transient errors:** 403/4xx fail fast; only 429/5xx and transient errors back off and retry.

In [ ]:
import os, json
from experiments import runner, report

# Tee every progress message to a persisted JSONL run-log so a full audit trail
# survives a disconnect. File IO is wrapped so logging can never crash the run.
RUN_LOG = os.path.join(RUN_DIR, 'run-log.jsonl')

def log_fn(msg):
    print(msg)
    try:
        with open(RUN_LOG, 'a', encoding='utf-8') as fh:
            fh.write(json.dumps({'msg': msg}) + '\n')
    except Exception:
        pass  # logging must never break the run

print('run-log ->', RUN_LOG)

# progress=log_fn  -> verbose, tee'd to JSONL.  resume=True -> reconnect-safe.
# show_progress=True -> tqdm bar per dataset/arm so a long run isn't a black box.
results = runner.run(
    cfg, seed=cfg.seeds[0], progress=log_fn, resume=True, show_progress=True
)

recs = results['records']
n_blocked = sum(1 for r in recs if r.get('blocked'))
n_err = sum(1 for r in recs if r.get('error') and not r.get('blocked'))
print(f"records: {len(recs)}  |  moderation-blocked: {n_blocked}  |  other errors: {n_err}")

### Per-record dump
Persist every individual record (arm, doc_id, question_id, answer_f1, routing, blocked, error) as JSONL — a flat, grep-able audit trail you can reload or diff offline.

In [ ]:
import os, json

RECORDS_PATH = os.path.join(RUN_DIR, 'records.jsonl')
n_written = 0
with open(RECORDS_PATH, 'w', encoding='utf-8') as fh:
    for r in results['records']:
        fh.write(json.dumps(r) + '\n')
        n_written += 1
print(f'wrote {n_written} records -> {RECORDS_PATH}')

## 8. Results: per-arm tables + AHC routing rate

**Fairness note — uniform drop of blocked questions.** Moderation runs on the *retrieved context*, which differs per arm, so a question can be blocked under one arm yet answered under another. Scoring it only where it survived would bias the comparison. So `aggregate` drops any question blocked (or errored) under **any** arm from **all** arms — every arm is scored on the identical question set. `report.blocked_report(...)` tells you how many questions were dropped.

In [ ]:
agg = report.aggregate(results['records'])            # uniform drop applied by default
routing = report.routing_rate(results['records'])
dropped = report.blocked_report(results['records'])   # per-dataset moderation-drop counts
print(report.to_markdown(agg, routing, dropped))
if dropped:
    print('\ndropped:', dropped)

## 9. Reading your numbers against the RAPTOR paper

Your reader is `gpt-oss-120b`, **not** the paper's GPT-4 / GPT-3 / UnifiedQA, so *absolute* numbers will not match a leaderboard — this is a **controlled arm-vs-arm** study (like the paper's own tables), where only the chunker changes. Use the paper as a sanity band, not a target. The question that matters: does `ahc` move Answer-F1 vs `token` on the **same docs, same reader**?

**QASPER — Answer token-F1 (RAPTOR), reference band:**

| Setting (RAPTOR + reader) | Answer-F1 |
|---|---:|
| RAPTOR + UnifiedQA | 36.6 |
| RAPTOR + GPT-3 | 53.1 |
| RAPTOR + GPT-4 | 55.7 |

> These are the paper's *reader-specific* headlines; `gpt-oss-120b` is none of them, so do not compare your absolute F1 to these directly. The signal here is the **`token` vs `ahc` delta on the same 5 docs** — and whether AHC's structure routing (section 6) translated into any movement at all.

## 10. Resuming after a disconnect

Recovery is automatic on two levels. **(1) LLM cache:** because `.llm_cache/` lives on Drive (section 2b), every already-completed call is served from disk on rerun — only missing calls hit the network. **(2) Run checkpoint:** the results file is rewritten after *every doc*, and `runner.run(..., resume=True)` skips any `(dataset, arm, doc)` already present. So after a reconnect, just re-run sections **1, 2, 2b, 5, 7, 8** — completed docs are skipped, not recomputed.

In [ ]:
# Nothing to copy if you mounted Drive in section 2b -- cache + results persist there.
# Sanity-check what has accumulated so far:
import os
print('cached LLM calls:', len(os.listdir(CACHE_DIR)) if os.path.isdir(CACHE_DIR) else 0)
print('results files:   ', os.listdir(RESULTS_DIR) if os.path.isdir(RESULTS_DIR) else [])